In [0]:
server = "capstone07dbserver.database.windows.net"
database = "capstoneproject07db"

username = "ltmadmin"
password = "P@55word"

jdbc_url = (
    f"jdbc:sqlserver://{server}:1433;"
    f"databaseName={database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

print("Azure SQL JDBC configuration completed.")

In [0]:
user_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.user_master")
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

print("Connection successful!")
print("Total records:", user_df.count())

display(user_df.limit(10))

In [0]:
# Read Category Lookup
category_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.document_category_lookup")
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

# Read Role Permission Lookup
role_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.role_permission_lookup")
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

print("User Master:", user_df.count())
print("Category Lookup:", category_df.count())
print("Role Permission:", role_df.count())

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS
capstone_project_07.bronze;

In [0]:
%sql
SHOW SCHEMAS IN capstone_project_07;

In [0]:
# Bronze table names
user_table = "capstone_project_07.bronze.bronze_user_master"

category_table = "capstone_project_07.bronze.bronze_document_category_lookup"

role_table = "capstone_project_07.bronze.bronze_role_permission_lookup"


# Write User Master
user_df.write.format("delta").mode("overwrite").saveAsTable(user_table)

# Write Category Lookup
category_df.write.format("delta").mode("overwrite").saveAsTable(category_table)

# Write Role Permission Lookup
role_df.write.format("delta").mode("overwrite").saveAsTable(role_table)

print("All three Azure SQL datasets successfully loaded into Bronze Delta.")

In [0]:
%sql
SELECT 'bronze_user_master' AS table_name, COUNT(*) AS total_records
FROM capstone_project_07.bronze.bronze_user_master

UNION ALL

SELECT 'bronze_document_category_lookup', COUNT(*)
FROM capstone_project_07.bronze.bronze_document_category_lookup

UNION ALL

SELECT 'bronze_role_permission_lookup', COUNT(*)
FROM capstone_project_07.bronze.bronze_role_permission_lookup;

In [0]:
%sql
SHOW TABLES IN capstone_project_07.bronze;

In [0]:
catalog = "capstone_project_07"
schema = "capstone_07_schema"

volume_path = "/Volumes/capstone_project_07/capstone_07_schema/capstone_07_ext_vol"

In [0]:
# ADLS source paths
access_path = f"{volume_path}/event_data/document_access_audit.csv"
search_path = f"{volume_path}/event_data/search_query_events.csv"

# Read CSV files without changing the original source values
access_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(access_path)
)

search_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(search_path)
)

print("Access Audit:", access_df.count())
print("Search Events:", search_df.count())

In [0]:
# Define Bronze table names
access_table = "capstone_project_07.bronze.bronze_document_access_audit"
search_table = "capstone_project_07.bronze.bronze_search_query_events"

# Write Access Audit
access_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(access_table)

# Write Search Events
search_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(search_table)



In [0]:
%sql
SELECT 'bronze_document_access_audit' AS table_name,
       COUNT(*) AS total_records
FROM capstone_project_07.bronze.bronze_document_access_audit

UNION ALL

SELECT 'bronze_search_query_events',
       COUNT(*)
FROM capstone_project_07.bronze.bronze_search_query_events;

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Source folder containing all 13 CDC batches
cdc_source_path = f"{volume_path}/cdc_data/document_metadata_changes"

# Persistent schema and checkpoint locations
schema_path = f"{volume_path}/_autoloader_schema/document_metadata_changes"
checkpoint_path = f"{volume_path}/_checkpoints/document_metadata_changes"

# Bronze target
cdc_table = "capstone_project_07.bronze.bronze_document_metadata_changes"

# Create supporting directories
dbutils.fs.mkdirs(schema_path)
dbutils.fs.mkdirs(checkpoint_path)

# Read CDC files using Auto Loader
cdc_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .option("inferSchema", "false")
    .load(cdc_source_path)
)

# Add ingestion metadata while preserving original columns
cdc_bronze_df = (
    cdc_stream_df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

# Write to Bronze Delta using a persistent checkpoint
cdc_query = (
    cdc_bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(cdc_table)
)

cdc_query.awaitTermination()

print("CDC Auto Loader ingestion completed.")
print("Bronze table:", cdc_table)

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM capstone_project_07.bronze.bronze_document_metadata_changes;

In [0]:
%sql
SELECT
    COUNT(DISTINCT _source_file) AS total_source_files,
    COUNT(*) AS total_records
FROM capstone_project_07.bronze.bronze_document_metadata_changes;

In [0]:
%sql
SELECT *
FROM capstone_project_07.bronze.bronze_document_metadata_changes
LIMIT 10;

In [0]:
%sql
SELECT 'bronze_user_master' AS table_name, COUNT(*) AS total_records
FROM capstone_project_07.bronze.bronze_user_master

UNION ALL

SELECT 'bronze_document_category_lookup', COUNT(*)
FROM capstone_project_07.bronze.bronze_document_category_lookup

UNION ALL

SELECT 'bronze_role_permission_lookup', COUNT(*)
FROM capstone_project_07.bronze.bronze_role_permission_lookup

UNION ALL

SELECT 'bronze_document_access_audit', COUNT(*)
FROM capstone_project_07.bronze.bronze_document_access_audit

UNION ALL

SELECT 'bronze_search_query_events', COUNT(*)
FROM capstone_project_07.bronze.bronze_search_query_events

UNION ALL

SELECT 'bronze_document_metadata_changes', COUNT(*)
FROM capstone_project_07.bronze.bronze_document_metadata_changes;

In [0]:
%sql
SELECT
    COUNT(DISTINCT _source_file) AS source_files,
    COUNT(*) AS total_records
FROM capstone_project_07.bronze.bronze_document_metadata_changes;